<img src="https://raw.githubusercontent.com/Brainchip-Inc/brainchip_devhub/main/docs/assets/0.-BC-dev-hub-LOGO-flicker.svg" alt="BrainChip Dev Hub" width="200"/>

# PlantVillage Disease Classification — Akida 2 Benchmark

Run Time: ~10 minutes

This notebook walks through evaluating and benchmarking a model on the **PlantVillage** dataset, targeting Akida 2 hardware.

For details on the dataset and preparation of the model, see the neighbouring README.md and plant_village_notebook_training.ipynb.

> **Note:** the hardware benchmark section needs a physical Akida 2 FPGA device. Everything is guarded, so the notebook runs end-to-end without a board — the hardware cells simply skip. It will not produce hardware latency numbers in Colab.

## Model

A pretrained Akida model is included in this repo, at `pretrained_models/mobilenet_plant_village_i8_w4_a4_qat.fbz`. As with all model files in the repo, that is handled via `git-lfs` (for efficient large file storage). If you haven't set that up yet, see the [Trained models](../../../README.md#trained-models) section of the top-level README.

If you have run through the training scripts or notebook, and prefer to benchmark the Akida model generated through that, simply point `MODELS_DIR` at `./models/` in the following cell.

In [1]:
import os
import numpy as np
import akida

from brainchip_utils.hardware_utils import AKIDA_CLOCKS_HZ

DATA_PATH = './data/plant_village'
MODELS_DIR = './pretrained_models/'
INPUT_SHAPE = (224, 224, 3)

# Which converted model to benchmark. The training pipeline produces two variants:
#   mobilenet_plant_village_i8_w8_a8.fbz     (8-bit)
#   mobilenet_plant_village_i8_w4_a4_qat.fbz (4-bit QAT)
MODEL_FILENAME = 'mobilenet_plant_village_i8_w4_a4_qat.fbz'
MODEL_FBZ = os.path.join(MODELS_DIR, MODEL_FILENAME)

# Clocks (see plant_village_benchmark.py and brainchip_utils.hardware_utils.AKIDA_CLOCKS_HZ).
MEASURED_CLOCK = AKIDA_CLOCKS_HZ['AKIDA2_FPGA']  # 25 MHz FPGA
PROJECTED_CLOCK = AKIDA_CLOCKS_HZ['AKD2500']     # 1 GHz target (pre-production)

To enable the following instructions: AVX2 AVX_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


To load the model, we simply pass the path to the `.fbz` file to the `akida.Model()` method:

In [2]:
ak_model = akida.Model(MODEL_FBZ)
ak_model.summary()

                 Model Summary                  
________________________________________________
Input shape    Output shape  Sequences  Layers
[224, 224, 3]  [1, 1, 38]    1          30    
________________________________________________

________________________________________________________________
Layer (type)                  Output shape    Kernel shape    

=========== SW/input_quantizer-dequantizer (Software) ==========

input_quantizer (Quantizer)   [224, 224, 3]   N/A             
________________________________________________________________
conv1 (InputConv2D)           [112, 112, 16]  (3, 3, 3, 16)   
________________________________________________________________
conv_dw_1 (DepthwiseConv2D)   [112, 112, 16]  (3, 3, 16, 1)   
________________________________________________________________
conv_pw_1 (Conv2D)            [112, 112, 32]  (1, 1, 16, 32)  
________________________________________________________________
conv_dw_2 (DepthwiseConv2D)   [56, 56, 32]    (3, 3

## Device

If an Akida 2 hardware device is connected it will be used for the latency benchmark; if not, the benchmark cells skip and only the software-backend accuracy/sparsity are reported. `get_akida_device` checks for a connected device of the correct IP version for this model (Akida 2) and returns `None` if none is present — so the whole notebook runs end-to-end without a board.

In [3]:
from brainchip_utils.hardware_utils import get_akida_device

device = get_akida_device(target_version=ak_model.ip_version)
if device is None:
    print('No compatible Akida hardware device found — benchmark will be skipped.')
else:
    print('Akida device found:', device)

Target Akida device found
Akida device found: <akida.core.HardwareDevice object at 0x7c4730c0b7f0>


## Samples

Akida is an event-driven architecture: computation scales with the number of non-zero activations, not with tensor size. That makes benchmark timings **input-dependent** — random or synthetic data would give artificially fast or slow results. We therefore load real images from the dataset to drive both the latency benchmark and the sparsity measurement.

In [4]:
from plant_village_data import get_samples

NUM_SAMPLES = 100
samples = get_samples(DATA_PATH, INPUT_SHAPE, num_samples=NUM_SAMPLES)

## Latency benchmark

`full_model_benchmark` from [brainchip_utils/hardware_utils.py](../../../brainchip_utils/hardware_utils.py) times the model on the connected device across mapping modes (`MapMode.Minimal`, fewest NPs, and `MapMode.AllNps`, maximum parallelism) so the latency/parallelism trade-off is visible.

The Akida 2 reference platform is an **FPGA clocked at 25 MHz**. Because the FPGA runs at a low clock, we also report a **projected latency** at a higher target clock: the cycle count is fixed for a given model and mapping regardless of clock rate, so the projection is an exact rescale of the measured cycles rather than an estimate. Power measurement is not included (the FPGA power path is still under development), so this benchmark is latency-only.

In [5]:
from brainchip_utils.hardware_utils import full_model_benchmark, get_mapping_stats

if device is not None:
    for mm in ['Minimal', 'AllNps', 'HwPr']:
        map_mode = getattr(akida.MapMode, mm)
        res = full_model_benchmark(ak_model, device, samples,
                                   map_mode=map_mode, clock_freq=MEASURED_CLOCK)
        projected_ms = res['mean_inf_clk'] / PROJECTED_CLOCK * 1000
        ak_model.map(device, mode=map_mode)
        num_nps, num_passes, num_sequences = get_mapping_stats(ak_model)
        print(f'[{mm}] NPs={num_nps} passes={num_passes} '
              f'latency@25MHz={res["mean_clk_ms"]:.3f} ms '
              f'projected@1000MHz={projected_ms:.3f} ms')
else:
    print('Hardware not available — skipping latency benchmark.')

Power measurement not available: only supported on AKD1500 (IpVersion.v1).
  Repeat 1/10 done.
  Repeat 2/10 done.
  Repeat 3/10 done.
  Repeat 4/10 done.
  Repeat 5/10 done.
  Repeat 6/10 done.
  Repeat 7/10 done.
  Repeat 8/10 done.
  Repeat 9/10 done.
  Repeat 10/10 done.

  Mean inference time:    348.211 ms  (σ=3.328 ms)
  Mean on-chip time:      344.832 ms  (8620799 clocks)
  Total inferences run:   1000
[Minimal] NPs=39 passes=2 latency@25MHz=344.832 ms projected@1000MHz=8.621 ms
Power measurement not available: only supported on AKD1500 (IpVersion.v1).
  Repeat 1/10 done.
  Repeat 2/10 done.
  Repeat 3/10 done.
  Repeat 4/10 done.
  Repeat 5/10 done.
  Repeat 6/10 done.
  Repeat 7/10 done.
  Repeat 8/10 done.
  Repeat 9/10 done.
  Repeat 10/10 done.

  Mean inference time:    347.278 ms  (σ=3.304 ms)
  Mean on-chip time:      343.899 ms  (8597471 clocks)
  Total inferences run:   1000
[AllNps] NPs=40 passes=2 latency@25MHz=343.899 ms projected@1000MHz=8.597 ms
Power measurement

## Per-layer benchmark & sparsity

`per_layer_benchmark` reconstructs latency layer by layer, and `compute_sparsity` measures per-layer activation sparsity. These are naturally correlated: because Akida processes events (non-zero activations), a layer's cost is proportional to its *input* sparsity — a layer receiving highly sparse inputs has far fewer events to process. Low-sparsity layers are therefore typically the latency bottlenecks, and this relationship is the direct signature of the event-driven compute model. Sparsity is computed on the software backend, so it is available even without a connected device.

In [6]:
from brainchip_utils.hardware_utils import per_layer_benchmark
from akida_models.sparsity import compute_sparsity
from brainchip_utils.plot_utils import pretty_print_sparsity

if device is not None:
    ak_model.map(device, mode=akida.MapMode.Minimal, hw_only=True)
    sparsity_dict = compute_sparsity(ak_model, samples=samples)
    pretty_print_sparsity(sparsity_dict)
    per_layer_results = per_layer_benchmark(ak_model, device, samples,
                                            repeats=NUM_SAMPLES, clock_freq=MEASURED_CLOCK)
    print('Per-layer benchmark complete.')
else:
    # Sparsity can still be computed on the software backend without hardware.
    sparsity_dict = compute_sparsity(ak_model, samples=samples)
    pretty_print_sparsity(sparsity_dict)
    print('Hardware not available — skipped latency; sparsity computed on software backend.')


Layer           Sparsity
------------------------
conv1            33.43%
conv_dw_1        25.24%
conv_pw_1        25.34%
conv_dw_2        13.12%
conv_pw_2        19.37%
conv_dw_3        35.26%
conv_pw_3        32.48%
conv_dw_4        17.42%
conv_pw_4        13.80%
conv_dw_5        39.03%
conv_pw_5        47.95%
conv_dw_6        25.86%
conv_pw_6        17.03%
conv_dw_7        46.10%
conv_pw_7        28.28%
conv_dw_8        39.49%
conv_pw_8        37.92%
conv_dw_9        39.75%
conv_pw_9        37.98%
conv_dw_10       38.81%
conv_pw_10       53.73%
conv_dw_11       45.46%
conv_pw_11       48.94%
conv_dw_12       36.66%
conv_pw_12       59.65%
conv_dw_13       49.55%
conv_pw_13       35.60%
predictions       0.03%
------------------------
Mean             33.69%

Layer               Latency (ms)       Clocks
-------------------------------------
input_quantizer           0.0000
conv1                     8.0631
conv_dw_1                24.3676
conv_pw_1                15.5361
conv_dw_2  